## `Neuropixels1ExtracellularLocations`

A Neuropixels 1.0 staggered layout (two sites per row, four columns). We use 96 sites here so the
probe is a comparable scale to the circuit (the default is 384 sites, i.e. a ~3.8 mm shank). Like
the linear array, it is inserted along the soma cloud's principal axis (as a probe would go down
the cortical depth), so it threads through the neurons rather than sitting across them. Every
electrode must map to a distinct position.

In [ ]:
from pathlib import Path

import bluepysnap as snap
import matplotlib.pyplot as plt
import numpy as np

import obi_one as obi
from obi_one.scientific.library.extracellular_locations import plot_extracellular_arrays

## Load the circuit and its soma positions

By default this uses the bundled tiny circuit; set `USE_STAGING_CIRCUIT = True` below to stage a circuit from staging by ID instead.

In [ ]:
# Optionally stage a circuit from staging by ID (as create_recording_array.py does) instead of the
# bundled tiny circuit. Set USE_STAGING_CIRCUIT = True (requires obi_auth + staging access); the
# resolved circuit's somas then drive both the visualisation and the electrode placement below.
USE_STAGING_CIRCUIT = True
# STAGING_CIRCUIT_ID = "ecc2e394-db99-4203-b162-608c0ef12cb9" # 10 neuron microcircuit
STAGING_CIRCUIT_ID = "1c81aa3e-7e9d-437a-91c8-aa5f7bc3bd02" # hex0


def load_staging_sonata_circuit() -> snap.Circuit:
    """Return the staging circuit's snap.Circuit, staging it into obi-output on first use.

    If the circuit is already staged locally under obi-output it is loaded straight from disk,
    without contacting staging (no token or download needed).
    """
    # Local obi-output cache location (outside the repo), as in create_recording_array.py.
    repo_root = next(
        base
        for base in [Path.cwd(), *Path.cwd().parents]
        if (base / "pyproject.toml").exists() and (base / "examples").exists()
    )
    obi_output_dir = repo_root.parent.parent / "obi-output"
    staged_config = (
        obi_output_dir
        / "entity_cache"
        / "sonata_circuit"
        / STAGING_CIRCUIT_ID
        / "circuit_config.json"
    )

    # Reuse the already-staged circuit if present; otherwise stage it from staging.
    if staged_config.exists():
        print(f"Using circuit already staged at {staged_config.parent}")
        return snap.Circuit(str(staged_config))

    from entitysdk import Client, ProjectContext
    from obi_auth import get_token

    from obi_one.utils import db_sdk

    db_client = Client(
        api_url="https://staging.openbraininstitute.org/api/entitycore",
        project_context=ProjectContext(
            virtual_lab_id=obi.LAB_ID_STAGING_TEST, project_id=obi.PROJECT_ID_STAGING_TEST
        ),
        token_manager=get_token(environment="staging"),
    )
    resolved_circuit, _ = db_sdk.resolve_circuit(
        obi.CircuitFromID(id_str=STAGING_CIRCUIT_ID),
        db_client=db_client,
        entity_cache=True,
        cache_root=obi_output_dir,
        temp_dir=obi_output_dir,
    )
    return resolved_circuit.sonata_circuit

In [ ]:
CIRCUIT_SUBPATH = Path("examples/data/tiny_circuits/N_10__top_nodes_dim6")


def resolve_repo_path(subpath: Path) -> Path:
    """Locate a repo-relative path by walking up from the current working directory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / subpath
        if candidate.exists():
            return candidate
    raise FileNotFoundError(subpath)


if USE_STAGING_CIRCUIT:
    circuit = load_staging_sonata_circuit()
else:
    circuit = snap.Circuit(str(resolve_repo_path(CIRCUIT_SUBPATH) / "circuit_config.json"))

print(circuit)

# Somas of the (non-virtual) biophysical population.
biophysical_pops = [
    name for name in circuit.nodes.population_names if circuit.nodes[name].type != "virtual"
]
soma_population = biophysical_pops[0]
soma_xyz = circuit.nodes[soma_population].get(properties=["x", "y", "z"]).to_numpy()
soma_centroid = soma_xyz.mean(axis=0)

# Principal axis of the soma cloud (used to orient a probe through the neurons).
_, _, principal_axes = np.linalg.svd(soma_xyz - soma_centroid, full_matrices=False)
soma_principal_axis = principal_axes[0]

print(f"Population {soma_population!r}: {len(soma_xyz)} somas")
print(f"Centroid (μm):   {np.round(soma_centroid, 1)}")
print(f"Principal axis:  {np.round(soma_principal_axis, 3)}")

## Helpers: place a rotated array, and point one along a direction

`build_probe(array_cls, origin, rotation, **params)` builds an array at `origin` (world x, y, z),
rotated by `rotation` = (`rotation_x`, `rotation_y`, `rotation_z`) degrees about the world axes.
`rotation_to_align_y(direction)` returns the angles that make the array's local `+Y` point along a
given direction (e.g. the soma cloud's principal axis) — so probes can follow the volume as before,
just expressed as angles.

In [ ]:
def build_probe(array_cls, origin, rotation=(0.0, 0.0, 0.0), **params):
    """Build an array at ``origin`` (x, y, z), rotated by ``rotation`` (rx, ry, rz) degrees.

    Returns the configured block and its world-coordinate electrode positions, shape ``(N, 3)``.
    """
    probe = array_cls(
        origin_x=float(origin[0]),
        origin_y=float(origin[1]),
        origin_z=float(origin[2]),
        rotation_x=float(rotation[0]),
        rotation_y=float(rotation[1]),
        rotation_z=float(rotation[2]),
        **params,
    )
    return probe, np.asarray(probe.get_global_electrode_xyz_locations(), dtype=float)


def rotation_to_align_y(direction):
    """Return (rx, ry, rz) degrees that point the array's local +Y along ``direction``.

    Solves ``Rz(rz) @ Rx(rx) @ (0, 1, 0) = direction`` with no roll (``rotation_y = 0``); valid for
    every direction, including the +/-Z poles.
    """
    d = np.asarray(direction, dtype=float)
    d = d / np.linalg.norm(d)
    rotation_x = np.degrees(np.arcsin(np.clip(d[2], -1.0, 1.0)))
    rotation_z = np.degrees(np.arctan2(-d[0], d[1]))
    return (float(rotation_x), 0.0, float(rotation_z))

## `LinearExtracellularLocations`

16 electrodes spaced 50 µm apart along the array's local `+Y` axis. We orient it along the soma
cloud's principal axis (like a probe inserted down the cortical depth), computed as rotation angles
via `rotation_to_align_y`. The measured inter-electrode distance always matches `spacing`, because
placement is a rigid rotation + translation.

In [ ]:
# Orient probes along the soma cloud's principal axis, expressed as rotation angles.
axis_rotation = rotation_to_align_y(soma_principal_axis)
print(
    f"Principal axis {np.round(soma_principal_axis, 3)} -> "
    f"rotation (deg) {tuple(round(a, 1) for a in axis_rotation)}"
)

linear_spacing = 50.0
linear_probe, linear_xyz = build_probe(
    obi.LinearExtracellularLocations,
    soma_centroid,
    rotation=axis_rotation,
    n_electrodes=16,
    spacing=linear_spacing,
)

# The probe axis follows the principal axis, and the spacing is preserved (rigid placement).
probe_axis = (linear_xyz[-1] - linear_xyz[0]) / np.linalg.norm(linear_xyz[-1] - linear_xyz[0])
step_distances = np.linalg.norm(np.diff(linear_xyz, axis=0), axis=1)
print(f"Probe axis:       {np.round(probe_axis, 3)}")
print(f"Measured spacing: {np.round(step_distances[:3], 3)} ... μm")
assert np.allclose(probe_axis, soma_principal_axis, atol=1e-6), "probe follows the principal axis"
assert np.allclose(step_distances, linear_spacing), "spacing preserved under rotation"

## `Neuropixels1ExtracellularLocations`

A Neuropixels 1.0 staggered layout (two sites per row, four columns). We use 96 sites so the probe
is a comparable scale to the circuit, orient it along the principal axis (the same `axis_rotation`),
and offset it laterally so it stays distinguishable from the linear probe in the combined view.
Every electrode maps to a distinct position.

In [ ]:
# Insert the Neuropixels probe along the principal axis (same axis_rotation as the linear one),
# offset laterally (along the soma cloud's 2nd principal axis) so it stays distinct.
lateral_axis = principal_axes[1]
lateral_extent = np.ptp((soma_xyz - soma_centroid) @ lateral_axis)
neuropixels_origin = soma_centroid + 0.3 * lateral_extent * lateral_axis
neuropixels_probe, neuropixels_xyz = build_probe(
    obi.Neuropixels1ExtracellularLocations,
    neuropixels_origin,
    rotation=axis_rotation,
    n_electrodes=96,
)

n_unique = len({tuple(np.round(position, 6)) for position in neuropixels_xyz})
print(f"Electrodes:        {len(neuropixels_xyz)}")
print(f"Unique positions:  {n_unique}")
assert n_unique == len(neuropixels_xyz), "every Neuropixels electrode must be at a distinct position"

### Effect of rotation

`rotation_x`, `rotation_y`, `rotation_z` rotate the array about the world X, Y and Z axes (in that
order), in degrees; all-zero leaves it along local `+Y`. Below we roll a Neuropixels probe about the
world Y axis (its long axis) and view it end-on (the X-Z cross-section): the staggered face turns in
place.

In [ ]:
roll_angles = [0.0, 30.0, 60.0, 90.0]
fig, axes = plt.subplots(1, len(roll_angles), figsize=(14, 4), sharex=True, sharey=True)
for ax, angle in zip(axes, roll_angles, strict=True):
    _, rolled_xyz = build_probe(
        obi.Neuropixels1ExtracellularLocations,
        (0.0, 0.0, 0.0),
        rotation=(0.0, angle, 0.0),
        n_electrodes=96,
    )
    ax.scatter(rolled_xyz[:, 0], rolled_xyz[:, 2], s=40, c="tab:red", alpha=0.6)
    ax.set_title(f"rotation_y = {angle:g}°")
    ax.set_xlabel("world X (μm)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("world Z (μm)")
fig.tight_layout()
plt.show()

## `GridExtracellularLocations` and `UTAHArrayExtracellularLocations`

Two planar (2D) arrays. `GridExtracellularLocations` is a configurable rectangular grid — here 6×10
with independent 300 µm (column) and 500 µm (row) spacing. `UTAHArrayExtracellularLocations` is the
classic Blackrock Utah array: a fixed 10×10 grid at 400 µm spacing. Both lie in the local X-Y plane;
we rotate the grid 90° about world X so it stands in the X-Z plane, distinct from the flat Utah
array, and centre both on the soma centroid.

In [ ]:
# Configurable grid (6 x 10, rectangular), stood up into the X-Z plane by a 90 deg rotation about X.
grid_probe, grid_xyz = build_probe(
    obi.GridExtracellularLocations,
    soma_centroid,
    rotation=(90.0, 0.0, 0.0),
    grid_rows=6,
    grid_columns=10,
    x_offset=300.0,
    y_offset=500.0,
)
print(f"Grid array electrodes: {len(grid_xyz)} ({grid_probe.grid_rows} x {grid_probe.grid_columns})")
assert len(grid_xyz) == grid_probe.grid_rows * grid_probe.grid_columns

# Fixed Utah array (10 x 10 at 400 um), flat in the X-Y plane, centred on the centroid.
utah_probe, utah_xyz = build_probe(
    obi.UTAHArrayExtracellularLocations,
    soma_centroid,
    rotation=(0.0, 0.0, 0.0),
)
print(f"Utah array electrodes: {len(utah_xyz)} (fixed 10 x 10 @ 400 um)")
assert len(utah_xyz) == 100
assert np.allclose(utah_xyz.mean(axis=0), soma_centroid), "centred grid -> centroid at origin"

## Combined figure

`plot_extracellular_arrays(circuit, electrode_locations)` takes a `Circuit` and a dictionary of extracellular-locations blocks and draws the three axis-plane projections, a 3D view, and one local-frame panel per array.

In [ ]:
electrode_locations = {
    "Linear": linear_probe,
    "Neuropixels": neuropixels_probe,
    "Grid": grid_probe,
    "Utah": utah_probe,
}
plot_extracellular_arrays(circuit, electrode_locations)
plt.show()

## Test endpoint. Service must be launched first (`make run-local`)

Calls the declared `/declared/extracellular-locations/block_summary` endpoint with a JSON
request body for an `ExtracellularLocationsUnion` member (a Neuropixels 1.0 array here), returning
the electrode positions in global coordinates.

In [ ]:
from entitysdk import Client, ProjectContext

from obi_auth import get_token

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=obi.LAB_ID_STAGING_TEST, project_id=obi.PROJECT_ID_STAGING_TEST)
db_client = Client(api_url="https://staging.openbraininstitute.org/api/entitycore", project_context=project_context, token_manager=token)


In [ ]:
import requests

# The service must be running (`make run-local`). Provide a valid bearer token, e.g. via
# `from obi_auth import get_token; token = get_token(environment="staging")`.
obi_one_api_url = "http://127.0.0.1:8100"
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

url = f"{obi_one_api_url}/declared/extracellular-locations/block_summary"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}
if virtual_lab_id:
    headers["virtual-lab-id"] = virtual_lab_id
if project_id:
    headers["project-id"] = project_id

# JSON request body: a Neuropixels 1.0 array (an ExtracellularLocationsUnion member).
request_body = {
    "type": "Neuropixels1ExtracellularLocations",
    "origin_x": 3695.0,
    "origin_y": -1089.0,
    "origin_z": -2797.0,
    "rotation_x": 0.0,
    "rotation_y": 0.0,
    "rotation_z": 30.0,
    "n_electrodes": 96,
}

response = requests.post(url, headers=headers, json=request_body)
if response.status_code == 200:
    summary = response.json()
    locations = summary["locations"]
    print(f"Success: {len(locations)} electrode locations")
    print("first electrode (μm):", locations[0])
    print("array properties:", {k: v for k, v in summary.items() if k != "locations"})
else:
    print(f"Error {response.status_code}: {response.text}")